# MemSwap on the logged outcomes: the running example end to end

This notebook follows the paper's running example through the architecture. Every mechanism answered every question
once and both judges scored every answer, so nothing here calls a model: the memory manager's read and the Mechanism
Evaluator's trial runs look up logged outcomes in `data/answers.csv`. It runs top to bottom in under a minute and needs
no API key.

The running example is a support agent. Its "order question" (*When did I place my last order?*) stands for the
temporal questions of LoCoMo, and its "promise question" (*What did you promise me last week?*) for the LongMemEval
questions about what the assistant said.

In [ ]:
import json, os, sys
from collections import Counter
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import evaluation as E, replay
from memswap.knowledge.knowledge_base import AdaptationGoals, MECHANISMS
from memswap.managed.memory_manager.question_classifier import LabelTypes
from memswap.loop import MemSwap
from memswap.plan.mechanism_selector import select
pd.set_option("display.precision", 3)
PURPLE, GREY, ORANGE = "#6B3FA0", "#666666", "#D08A2E"      # the colours of the architecture figure

## 1. The data
`answers.csv` has one row per question and system: the verdict of each judge and the memory tokens.

In [ ]:
answers, streams, q, S_by_judge, _, TOK = E.load()
S = S_by_judge["j1"]                                         # verdicts of the primary judge, one column per system
texts = json.load(open(os.path.join(ROOT, "data", "q_texts.json")))
print(len(q), "questions;", answers.system.nunique(), "systems")
q.groupby("dataset").agg(questions=("question_type", "size"), conversations=("conversation_id", "nunique"), question_types=("question_type", "nunique"))

## 2. Monitor: the Mechanism Evaluator profiles the first labelled batch
We hold out one LoCoMo conversation. The architect supplies the other nine as the first labelled batch. The Mechanism
Evaluator runs all three reading mechanisms on every question of the batch and stores, per question type, the accuracy
$a_t(m)$, the memory tokens $c_t(m)$ and the number of questions $n_t$ in the mechanism profiles.

In [ ]:
labels = LabelTypes(q)                                       # the benchmark label gives the question type here
loco = q[q.dataset == "locomo"]
held_out = "conv-26"
first_batch = loco.index[loco.conversation_id != held_out].tolist()
read = lambda m, qid: (S.at[qid, m], TOK.at[qid, m])         # the logged outcome of mechanism m on a question
ms = MemSwap(AdaptationGoals(margin=0.01, cap=0), labels, read, S, TOK)
ms.first_batch("demo", first_batch, labels.classify)
est = ms.kb.estimates.batch["demo"]
rows = [{"question type": t.split("|")[1], "n": r["n"], **{m + " acc.": r["accuracy"][m] for m in MECHANISMS},
         **{m + " tokens": round(r["mean_tokens"][m]) for m in MECHANISMS}} for t, r in sorted(est.items())]
pd.DataFrame(rows).set_index("question type")

## 3. Plan: the Mechanism Selector applies the selection rule
A mechanism is *sufficient* for a question type if its cautious estimate, the one-sided 95% Wilson lower bound
$L_t(m)$, is at most the accuracy margin $\delta$ below the best accuracy. The rule picks the cheapest sufficient
mechanism, and keeps the best one if none is sufficient.

In [ ]:
r = est["locomo|temporal"]                                   # the order question's question type
fig, ax = plt.subplots(figsize=(5.2, 2.6))
x = np.arange(3)
ax.bar(x, [100 * r["accuracy"][m] for m in MECHANISMS], color=[GREY, ORANGE, PURPLE], width=0.55, label="measured accuracy")
ax.scatter(x, [100 * r["wilson_lb"][m] for m in MECHANISMS], color="black", marker="_", s=600, zorder=3, label="Wilson lower bound")
for d, ls in ((1, "--"), (10, ":")):
    ax.axhline(100 * max(r["accuracy"].values()) - d, color="black", lw=0.8, ls=ls, label=f"best minus {d} point" + ("s" if d > 1 else ""))
ax.set_xticks(x, [f"{m.replace('_', ' ')}\n{r['mean_tokens'][m]:,.0f} tokens" for m in MECHANISMS]); ax.set_ylim(60, 90); ax.set_ylabel("accuracy (%)")
ax.set_title(f"temporal questions, {r['n']} profiled"); ax.legend(fontsize=7, ncol=2, frameon=False); plt.tight_layout(); plt.show()
for margin in (0.01, 0.10):
    k = {m: round(r["accuracy"][m] * r["n"]) for m in MECHANISMS}
    print(f"margin {round(100 * margin):>2} point(s): the rule selects", select(k, r["n"], r["mean_tokens"], margin)["chosen"])

At a one-point margin no cheaper mechanism is sufficient, so the rule keeps the most accurate one, single lookup. At a
ten-point margin planned read becomes sufficient and the rule selects it. The Plan Enactor then sends the routing
table $R$ to the Router.

In [ ]:
pd.Series({t.split("|")[1]: m for t, m in ms.kb.routing.table["demo"].items()}, name="R (mechanism per question type)").to_frame()

## 4. The request path: one question through the memory manager
The Question Classifier gives the question a question type, the Router looks it up in $R$, and the mechanism reads the
store and answers. The managing system is not involved.

In [ ]:
held = loco[(loco.conversation_id == held_out) & (loco.question_type == "temporal")]
qid = held.index[0]
rec = ms.handle(qid, "demo", 1)
logged = {}
for line in open(os.path.join(ROOT, "data", "answer_texts.jsonl")):
    a = json.loads(line)
    if a["question_id"] == qid and a["system"] == rec["mechanism"]:
        logged = a
print("question      :", texts[qid]); print("question type :", rec["type"]); print("mechanism     :", rec["mechanism"])
print("answer        :", logged["answer"][:200]); print("gold answer   :", logged["gold_answer"])
print("judged correct:", bool(rec["outcome"]), "| memory tokens:", int(rec["tokens"]))

## 5. The architect lowers the token budget
During a sale the team lowers the budget $B$. The Cost Analyzer flags the change, the Plan Optimizer picks the most
accurate plan whose expected cost fits $B$, and the Plan Enactor sends the new routing table.

In [ ]:
before = dict(ms.kb.routing.table["demo"])
moves = ms.set_budget(3168.0, at=2)                          # the lowered budget of the paper's drop scenario
after = ms.kb.routing.table["demo"]
display(pd.DataFrame({"before": before, "B = 3,168": after}).rename(index=lambda t: t.split("|")[1]))
print("next order question is answered with:", ms.handle(held.index[1], "demo", 2)["mechanism"])

The whole drop scenario on LoCoMo (all ten folds): the budget drops by 40% halfway and returns at three quarters.

In [ ]:
order = [(i, loco.at[i, "conversation_id"]) for i in loco.index]
folds = {c: loco.index[loco.conversation_id != c].tolist() for c in sorted(loco.conversation_id.unique())}
fixed_tok, levels = E.settings_for("locomo", order, q, S, TOK)
base, n = levels[3], len(order)
run = replay.run_budget(order, q, S, TOK, folds, None, lambda i: base * 0.6 if n // 2 <= i < 3 * n // 4 else base)
W = run["windows"]; ends = [w["end"] for w in W]
fig, ax = plt.subplots(figsize=(6.4, 2.4))
ax.bar([e - 25 for e in ends], [w["mean"] for w in W], width=43, color=[PURPLE if w["kept"] else "#C9B8E3" for w in W])
ax.step([0] + ends, [W[0]["B"]] + [w["B"] for w in W], where="pre", color="black", ls="--", lw=1, label="token budget B")
for rp in run["replans"]:
    ax.axvline(rp["at"], color=GREY, ls=":", lw=0.8)
ax.set_xlabel("questions served"); ax.set_ylabel("window mean (memory tokens)"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()
print("re-plans:", [(rp["at"], rp["why"]) for rp in run["replans"]])
print(f"accuracy {100 * run['memswitch']['accuracy']:.2f}% at {run['memswitch']['mean_memory_tokens']:,.0f} tokens per question; "
      f"{sum(w['kept'] for w in W)} of {len(W)} windows within the budget (light bars are over it)")

## 6. The workload changes
Stream 3 profiles all of LoCoMo and then replays LongMemEval-R, whose question types are new. The Workload Analyzer
flags them, and the Router answers them with the default mechanism $m_0$. A mechanism fixed at design time has no such
fallback.

In [ ]:
g = streams[streams.stream == 3]
fb = g[g.role == "first_batch"].question_id.tolist(); sq = g[g.role == "stream"].sort_values("position").question_id.tolist()
out = replay.run_stream(fb, sq, q, S, S, TOK, 0.01, 0, "demo")
print("default mechanism m0:", out["default_mechanism"]); print("new question types flagged:", [t.split("|")[1] for t in out["types_in_stream_not_in_first_batch"]])
acc = {"fixed single lookup": 100 * S.loc[sq, "single_lookup"].mean(), "fixed planned read": 100 * S.loc[sq, "planned_read"].mean(),
       "fixed iterative search": 100 * S.loc[sq, "iterative_search"].mean(), "MemSwap": 100 * out["memswitch"]["accuracy"]}
fig, ax = plt.subplots(figsize=(5.2, 2.2))
ax.barh(list(acc), list(acc.values()), color=[GREY, GREY, GREY, PURPLE]); ax.set_xlabel("accuracy on the replayed questions (%)"); ax.set_xlim(50, 90)
for i, v in enumerate(acc.values()): ax.text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()
print(f"on LoCoMo, fixed single lookup answered {100 * S.loc[loco.index, 'single_lookup'].mean():.1f}%")

## Where to go next
- `python reproduce.py` rebuilds every table and number of the paper and compares them with `results/paper_numbers.csv`.
- `python stats/verify_stats.py` recomputes the statistics without our code.
- `docs/EXTEND.md` shows how to add a reading mechanism, a dataset, or another selection rule or planner.